In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

def summarize_df(df, file_label, name_col=None, include_messages=False, list_limit=50):
    row_count = len(df)
    cmdb_vals = df['cmdb_id'].dropna().unique()[:list_limit].tolist()
    # name fields (kpi/trace/log)
    if name_col and name_col in df.columns:
        name_vals = df[name_col].dropna().unique()[:list_limit].tolist()
    else:
        name_vals = []
    # timestamps -> timezone aware Asia/Shanghai
    if 'timestamp' in df.columns and not df['timestamp'].dropna().empty:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        earliest = ts.min().strftime('%Y-%m-%d %H:%M:%S %Z')
        latest = ts.max().strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest = None
        latest = None
    extra = None
    if include_messages and 'message' in df.columns:
        top_msgs = df['message'].dropna().value_counts().head(5).index.tolist()
        extra = top_msgs
    return {
        'file': file_label,
        'rows': row_count,
        'distinct_cmdb_ids': cmdb_vals,
        'distinct_name_fields': name_vals,
        'earliest_ts': earliest,
        'latest_ts': latest,
        'top_messages': extra
    }

summaries = [
    summarize_df(metric, 'metric.csv', name_col='kpi_name'),
    summarize_df(trace, 'trace.csv', name_col='trace_name'),
    summarize_df(log, 'log.csv', name_col='log_name'),
    summarize_df(error_logs, 'error_logs.csv', include_messages=True)
]

summary_df = pd.DataFrame(summaries)

# Keep output compact: show up to first 4 rows (we only have 4), and select columns
summary_df = summary_df[['file','rows','distinct_cmdb_ids','distinct_name_fields','earliest_ts','latest_ts','top_messages']]

summary_df

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv: 22,323 rows; sample distinct cmdb_ids: [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01, ...]; "
    "sample kpi names: [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM-Memory_7778_JVM_Memory_HeapMemoryUsage, ...]; "
    "time range: 2021-03-23 23:00:00 CST — 2021-03-23 23:30:00 CST.\n\n"
    "trace.csv: 6,906 rows; sample distinct cmdb_ids: [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, ...]; "
    "sample trace names: [trace.self.duration_mean, trace.self.duration_p50, ...]; "
    "time range: 2021-03-23 23:00:00 CST — 2021-03-23 23:30:00 CST.\n\n"
    "log.csv: 388 rows; sample distinct cmdb_ids: [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, ...]; "
    "sample log names: [log.error_count, log.row_count, log.gc_pause_..., ...]; "
    "time range: 2021-03-23 23:00:00 CST — 2021-03-23 23:30:00 CST.\n\n"
    "error_logs.csv: 0 rows; no cmdb_ids, no timestamps, and no messages present."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file   rows                                  distinct_cmdb_ids                               distinct_name_fields              earliest_ts                latest_ts top_messages
0      metric.csv  22323  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...  [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM...  2021-03-23 23:00:00 CST  2021-03-23 23:30:00 CST         None
1       trace.csv   6906  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [trace.self.duration_mean, trace.self.duration...  2021-03-23 23:00:00 CST  2021-03-23 23:30:00 CST         None
2         log.csv    388  [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apach...  [log.error_count, log.row_count, log.gc_pause_...  2021-03-23 23:00:00 CST  2021-03-23 23:30:00 CST         None
3  error_logs.csv      0                                                 []                                                 []                     None                     None           []```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric if available, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Components of interest
components = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02',
    'Mysql01','Mysql02','Redis01','Redis02'
]

# Filter to allowed components and ensure numeric values
df = metric[metric['cmdb_id'].isin(components)].copy()
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Group across entire file (no time filtering) and compute required stats
grp = df.groupby(['cmdb_id','kpi_name'])['value']
agg_df = grp.agg(
    count='count',
    p95=lambda s: s.quantile(0.95),
    p90=lambda s: s.quantile(0.90),
    p50=lambda s: s.quantile(0.50),
    p5 =lambda s: s.quantile(0.05),
    mean='mean',
    stddev='std'
).reset_index()

# Sort by P95 descending and prepare compact output
agg_df = agg_df.sort_values('p95', ascending=False)
agg_df = agg_df[['cmdb_id','kpi_name','count','p95','p90','p50','p5','mean','stddev']]

# Round numeric columns for compactness
num_cols = ['p95','p90','p50','p5','mean','stddev']
agg_df[num_cols] = agg_df[num_cols].round(6)

# Results: total distinct component-KPI combos and top 20 by p95
total_combos = agg_df.shape[0]
top20 = agg_df.head(20)

total_combos, top20

```
Out[2]:
```
summary = (
    "Summary of aggregation over metric.csv (entire file):\n\n"
    "• Total distinct component–KPI combinations detected: 1,114.\n\n"
    "• Top contributors by global P95 (highest values):\n"
    "  - Mysql02: 'Mysql-MySQL_3306_Innodb data written' — p95 ≈ 2.767982e+12 (count=31).\n"
    "  - Mysql02: 'Mysql-MySQL_3306_Innodb buffer pool read requests' — p95 ≈ 1.665285e+12 (count=31).\n"
    "  - Mysql02: 'Mysql-MySQL_3306_Innodb os log written' — p95 ≈ 7.254525e+11 (count=31).\n"
    "  - Mysql01: 'Mysql-MySQL_3306_Innodb data written' — p95 ≈ 6.251288e+11 (count=31).\n"
    "  - Mysql02: 'Mysql-MySQL_3306_Rows Read' — p95 ≈ 5.444316e+11 (count=31).\n\n"
    "• Notable pattern: Several Tomcat JVM memory KPIs (Tomcat01/02/03/04) show identical, large constant values\n"
    "  (p95 ≈ 4.187619e+09, count=6) with stddev = 0, indicating those metrics are constant in the recorded samples.\n\n"
    "• The returned top-20 by P95 is dominated by Mysql-related I/O and row/bytes metrics (high magnitudes),\n"
    "  followed by a cluster of constant Tomcat JVM memory metrics.\n\n"
    "If you want, I can export the top-20 table (cmdb_id, kpi_name, count, p95, p90, p50, p5, mean, stddev) for closer inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(1114,        cmdb_id                                           kpi_name  count           p95           p90           p50            p5          mean        stddev
441    Mysql02               Mysql-MySQL_3306_Innodb data written     31  2.767982e+12  2.767935e+12  2.767754e+12  9.408378e+04  1.517864e+12  1.400208e+12
430    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests     31  1.665285e+12  1.665285e+12  1.665284e+12  9.339416e+02  9.132206e+11  8.424332e+11
451    Mysql02             Mysql-MySQL_3306_Innodb os log written     31  7.254525e+11  7.254307e+11  7.253354e+11  7.373653e+04  3.977977e+11  3.669627e+11
304    Mysql01               Mysql-MySQL_3306_Innodb data written     31  6.251288e+11  6.251123e+11  6.250422e+11  4.061013e+04  3.427889e+11  3.162179e+11
481    Mysql02                         Mysql-MySQL_3306_Rows Read     31  5.444316e+11  5.444316e+11  5.444315e+11  8.128335e+01  2.985592e+11  2.754167e+11
412    Mysql02                 Mysql-MySQL_3306_Handler Read Prev     31  5.442457e+11  5.442457e+11  5.442456e+11  5.872500e+01  2.984573e+11  2.753227e+11
389    Mysql02                        Mysql-MySQL_3306_Bytes Sent     31  1.853260e+11  1.853207e+11  1.852969e+11  1.909381e+04  1.016225e+11  9.374535e+10
252    Mysql01                        Mysql-MySQL_3306_Bytes Sent     31  7.787292e+10  7.787133e+10  7.786459e+10  5.429559e+03  4.270224e+10  3.939221e+10
438    Mysql02                  Mysql-MySQL_3306_Innodb data read     31  3.317284e+10  3.317284e+10  3.317284e+10  0.000000e+00  1.819156e+10  1.678146e+10
314    Mysql01             Mysql-MySQL_3306_Innodb os log written     31  9.144968e+09  9.144739e+09  9.143752e+09  7.808000e+02  5.014653e+09  4.625946e+09
949   Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
947   Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
706   Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
708   Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
825   Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
1071  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
1069  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
827   Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
277    Mysql01             Mysql-MySQL_3306_Handler Read Rnd Next     31  3.602009e+09  3.601926e+09  3.601566e+09  1.509250e+02  1.975173e+09  1.822069e+09
344    Mysql01                         Mysql-MySQL_3306_Rows Read     31  3.255939e+09  3.255862e+09  3.255524e+09  1.303416e+02  1.785400e+09  1.647006e+09)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse metric if available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Allowed components
components = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02',
    'Mysql01','Mysql02','Redis01','Redis02'
]

# Prepare dataframe limited to allowed components and ensure numeric values
df_all = metric[metric['cmdb_id'].isin(components)].copy()
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
# create tz-aware timestamp in Asia/Shanghai
df_all['ts'] = pd.to_datetime(df_all['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global thresholds (p95 and p5) across entire file per component-KPI
thresholds = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
    p95=lambda s: s.quantile(0.95),
    p5 =lambda s: s.quantile(0.05)
).reset_index()

# 2) Filter rows to incident window (inclusive)
start = pd.Timestamp('2021-03-23 23:00:00', tz=tz)
end   = pd.Timestamp('2021-03-23 23:30:00', tz=tz)
df_window = df_all[(df_all['ts'] >= start) & (df_all['ts'] <= end)].copy()

# Merge thresholds into window rows to evaluate anomalies
merged = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Flag anomalies in window using global thresholds
merged['is_anom'] = (merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])

# 3) Aggregate per component-KPI in the window
def earliest_anom_ts(s_ts, s_is_anom):
    if s_is_anom.any():
        ts = s_ts[s_is_anom].min()
        return ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    return None

grp = merged.groupby(['cmdb_id','kpi_name'], as_index=False)

agg = grp.agg(
    count_in_window = ('value','count'),
    anomalies_count = ('is_anom','sum'),
    earliest_anom_ts = ('ts', lambda s: None),  # placeholder
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# fill earliest_anom_ts by group (need access to is_anom)
earliest_map = {}
for (c,k), sub in merged.groupby(['cmdb_id','kpi_name']):
    earliest_map[(c,k)] = earliest_anom_ts(sub['ts'], sub['is_anom'])
agg['earliest_anom_ts'] = agg.apply(lambda r: earliest_map.get((r['cmdb_id'], r['kpi_name'])), axis=1)

# Compute relative deviations and direction
def compute_dev(row):
    p95 = row['p95']
    p5  = row['p5']
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    # safe computations
    high_dev = np.nan
    low_dev = np.nan
    if pd.notna(p95) and p95 != 0:
        high_dev = (maxv - p95) / p95
    if pd.notna(p5) and p5 != 0:
        low_dev = (p5 - minv) / p5
    # choose the larger positive breach
    # If both NaN, set 0
    hi = high_dev if pd.notna(high_dev) else -np.inf
    lo = low_dev if pd.notna(low_dev) else -np.inf
    if hi >= lo:
        direction = 'high'
        max_rel = hi if hi != -np.inf else np.nan
    else:
        direction = 'low'
        max_rel = lo if lo != -np.inf else np.nan
    return pd.Series({'direction': direction, 'max_relative_deviation': max_rel})

devs = agg.apply(compute_dev, axis=1)
agg = pd.concat([agg, devs], axis=1)

# 4) Filter combos that have anomalies_count > 0
anoms = agg[agg['anomalies_count'] > 0].copy()

# Prepare final compact table and sorting by max_relative_deviation desc
cols = ['cmdb_id','kpi_name','count_in_window','anomalies_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','direction','max_relative_deviation']
anoms = anoms[cols]

# Round numeric columns for compactness
for c in ['max_value_in_window','min_value_in_window','p95','p5','max_relative_deviation']:
    anoms[c] = pd.to_numeric(anoms[c], errors='coerce')
anoms[['p95','p5','max_relative_deviation']] = anoms[['p95','p5','max_relative_deviation']].round(6)
anoms[['max_value_in_window','min_value_in_window']] = anoms[['max_value_in_window','min_value_in_window']].round(6)

anoms_sorted = anoms.sort_values('max_relative_deviation', ascending=False).head(20)

total_anomalous_combos = anoms.shape[0]

# Final compact output: total count and top 20 rows
total_anomalous_combos, anoms_sorted.reset_index(drop=True)

```
Out[3]:
```
- Total number of component–KPI combos with anomalies in the incident window: 1,114.

- Top findings (top 20 sorted by max_relative_deviation, compact highlights):
  1. Tomcat03 — Tomcat-Threads..."CurrentThreads" (count_in_window=31, anomalies=31). Earliest anomaly: 2021-03-23 23:00:00 CST. max=2, min=0, p95=0.5, p5=0.0, direction=high, max_relative_deviation=3.0 (largest relative high-side breach).
  2. Tomcat04 — OSLinux CPU single-core idle (count=15, anomalies=15). Earliest anomaly: 2021-03-23 23:01:00 CST. max≈0.332281, p95≈0.099684, direction=high, max_relative_deviation≈2.333333.
  3. MG01 — local disk DSKRead (count=15, anomalies=15). Earliest: 2021-03-23 23:01:00 CST. max≈245.6, p95≈73.68, direction=high, max_relative_deviation≈2.333333.
  4–9. Several MG01/Tomcat/MG02 disk I/O KPIs (DSKBps, DSKRTps, DSKPer..., DSKRead) show identical high-side breach pattern (many anomalies, earliest ≈23:01, direction=high, max_relative_deviation≈2.333333).
  10. Tomcat03 — large DSKRead spike (count=15, anomalies=13). earliest 23:01, max≈286.53, p95≈90.16, max_rel≈2.178.
  11–13. Tomcat04 and Tomcat03 CPU WIO metrics and disk RTps also show high-side breaches (various counts; earliest anomalies mostly 23:01–23:18).
  14–16. Disk write metrics on Tomcat03 and MG01 show notable high-side deviations (max_rel ≈1.29 and ≈1.21).
  16–17. Tomcat02 DSKBps and Mysql02 CurrentSQLMaxRunningTime show high-side breaches (Mysql02: count=31, anomalies=31, max_rel=1.0).
  18–19. A few low-direction cases appear in the top-20: Tomcat02 single-core idle (direction=low; earliest 23:05; max_rel=1.0) and Mysql02 Row Lock Waits (low; earliest 23:06; max_rel=1.0).

- Patterns and interpretation:
  - The majority of top anomalies are high-side breaches (CPU idle falling is a low-side breach). Disk I/O and per-disk metrics across MG01/MG02 and Tomcat nodes show consistent, substantial high-side deviations during the window (earliest anomalies cluster around 23:00–23:01).
  - Some MySQL metrics (e.g., CurrentSQLMaxRunningTime, Row Lock Waits) also show full-window anomalies (anomalies_count = full sample count) indicating sustained deviations in the window.
  - The largest relative deviation observed among top rows is 3.0 (Tomcat03 threads), with many disk/IO metrics around 2.33 and several others >1.0.

If you want, I can produce the full top-20 table as CSV-like rows or filter to a specific component to inspect raw anomaly timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(1114,      cmdb_id                                           kpi_name  count_in_window  anomalies_count         earliest_anom_ts  max_value_in_window  min_value_in_window            p95         p5 direction  max_relative_deviation
0   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31               31  2021-03-23 23:00:00 CST             2.000000               0.0000       0.500000   0.000000      high                3.000000
1   Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle               15               15  2021-03-23 23:01:00 CST             0.332281               0.0000       0.099684   0.000000      high                2.333333
2       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               15               15  2021-03-23 23:01:00 CST           245.600000               0.0000      73.680000   0.000000      high                2.333333
3       MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               15               15  2021-03-23 23:01:00 CST             2.000000               0.0000       0.600000   0.000000      high                2.333333
4       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               15               15  2021-03-23 23:01:00 CST             3.266700               0.0000       0.980010   0.000000      high                2.333333
5   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               15               15  2021-03-23 23:01:00 CST             0.216667               0.0000       0.065000   0.000000      high                2.333333
6       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...               15               15  2021-03-23 23:01:00 CST             0.200000               0.0000       0.060000   0.000000      high                2.333333
7   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               15               15  2021-03-23 23:01:00 CST             6.000000               0.0000       1.800000   0.000000      high                2.333333
8   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...               15               15  2021-03-23 23:00:00 CST             0.200000               0.0000       0.060000   0.000000      high                2.333333
9       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               15               15  2021-03-23 23:01:00 CST             3.000000               0.0000       0.900000   0.000000      high                2.333333
10  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead               15               13  2021-03-23 23:01:00 CST           286.533333               0.0000      90.160000   0.000000      high                2.178054
11  Tomcat04                             OSLinux-CPU_CPU_CPUWio               31                4  2021-03-23 23:18:00 CST             1.010101               0.0166       0.336639   0.020782      high                2.000547
12  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps               15               13  2021-03-23 23:01:00 CST             3.533333               0.0000       1.188333   0.000000      high                1.973353
13  Tomcat03                             OSLinux-CPU_CPU_CPUWio               31                4  2021-03-23 23:04:00 CST             0.611100               0.0166       0.264030   0.018706      high                1.314505
14  Tomcat03   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite               15                3  2021-03-23 23:05:00 CST            66.666667               7.7333      29.146667   7.733300      high                1.287283
15      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite               15                2  2021-03-23 23:03:00 CST            55.466700               7.0667      25.133343   7.346700      high                1.206897
16  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               15               12  2021-03-23 23:01:00 CST             8.000000               0.0000       3.800000   0.000000      high                1.105263
17   Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime               31               31  2021-03-23 23:00:00 CST             1.000000               0.0000       0.500000   0.000000      high                1.000000
18  Tomcat02                    OSLinux-CPU_CPU-1_SingleCpuidle               15                2  2021-03-23 23:05:00 CST            99.418000               0.0000      99.091380  46.686080       low                1.000000
19   Mysql02             Mysql-MySQL_3306_Innodb Row Lock Waits               31                4  2021-03-23 23:06:00 CST        799752.000000               0.0000  799740.500000   0.008200       low                1.000000)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse tz and components if available, otherwise define
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

try:
    components
except NameError:
    components = [
        'apache01','apache02',
        'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
        'MG01','MG02','IG01','IG02',
        'Mysql01','Mysql02','Redis01','Redis02'
    ]

# reuse trace if available, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# 1) Filter to allowed components and ensure numeric values
df_all = trace[trace['cmdb_id'].isin(components)].copy()
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['ts'] = pd.to_datetime(df_all['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute global thresholds (p95 and p5) across entire file per (cmdb_id, trace_name)
thresholds = df_all.groupby(['cmdb_id','trace_name'])['value'].agg(
    p95=lambda s: s.quantile(0.95),
    p5 =lambda s: s.quantile(0.05)
).reset_index()

# 2) Filter rows to the incident window (inclusive)
start = pd.Timestamp('2021-03-23 23:00:00', tz=tz)
end   = pd.Timestamp('2021-03-23 23:30:00', tz=tz)
df_window = df_all[(df_all['ts'] >= start) & (df_all['ts'] <= end)].copy()

# Merge thresholds into window rows
merged = df_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies in window using global thresholds
merged['is_anom'] = (merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])

# Helper to get earliest anomaly ts per group
def earliest_anom_ts(sub):
    anoms = sub[sub['is_anom']]
    if anoms.shape[0] == 0:
        return None
    return anoms['ts'].min().strftime('%Y-%m-%d %H:%M:%S %Z')

# 3) Aggregate per component-trace in the window
grp = merged.groupby(['cmdb_id','trace_name'], as_index=False)

agg = grp.agg(
    count_in_window = ('value','count'),
    anomalies_count = ('is_anom','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# compute earliest_anom_ts by group iteration
earliest_map = {}
for (c,tname), sub in merged.groupby(['cmdb_id','trace_name']):
    earliest_map[(c,tname)] = earliest_anom_ts(sub)
agg['earliest_anom_ts'] = agg.apply(lambda r: earliest_map.get((r['cmdb_id'], r['trace_name'])), axis=1)

# Compute direction and max_relative_deviation safely
def compute_dev(row):
    p95 = row['p95']
    p5  = row['p5']
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    # initialize as NaN
    high_dev = np.nan
    low_dev = np.nan
    if pd.notna(p95) and p95 != 0:
        high_dev = (maxv - p95) / p95
    if pd.notna(p5) and p5 != 0:
        low_dev = (p5 - minv) / p5
    # treat negative deviations as 0 (we care about breaches)
    high_dev = high_dev if pd.notna(high_dev) and high_dev > 0 else (np.nan if pd.isna(high_dev) else 0.0)
    low_dev  = low_dev  if pd.notna(low_dev)  and low_dev  > 0 else (np.nan if pd.isna(low_dev)  else 0.0)
    # choose larger
    hi = high_dev if pd.notna(high_dev) else -np.inf
    lo = low_dev if pd.notna(low_dev) else -np.inf
    if hi >= lo:
        direction = 'high'
        max_rel = hi if hi != -np.inf else np.nan
    else:
        direction = 'low'
        max_rel = lo if lo != -np.inf else np.nan
    return pd.Series({'direction': direction, 'max_relative_deviation': max_rel})

devs = agg.apply(compute_dev, axis=1)
agg = pd.concat([agg, devs], axis=1)

# 4) Filter combos that have anomalies_count > 0
anoms = agg[agg['anomalies_count'] > 0].copy()

# select and round for compactness
cols = ['cmdb_id','trace_name','count_in_window','anomalies_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','direction','max_relative_deviation']
anoms = anoms[cols]

# numeric rounding
for c in ['max_value_in_window','min_value_in_window','p95','p5','max_relative_deviation']:
    anoms[c] = pd.to_numeric(anoms[c], errors='coerce')
anoms[['p95','p5','max_relative_deviation']] = anoms[['p95','p5','max_relative_deviation']].round(6)
anoms[['max_value_in_window','min_value_in_window']] = anoms[['max_value_in_window','min_value_in_window']].round(6)

anoms_sorted = anoms.sort_values('max_relative_deviation', ascending=False).head(20)

total_anomalous_combos = anoms.shape[0]

# Final compact output: total count and top 20 rows
total_anomalous_combos, anoms_sorted.reset_index(drop=True)

```
Out[4]:
```
summary = (
    "Trace anomaly summary (window 2021-03-23 23:00:00–23:30:00 CST):\n\n"
    "• Total component–trace combinations with anomalies: 168.\n\n"
    "• Top anomalous traces by relative deviation (high-side dominates):\n"
    "  1) Tomcat04 — trace.self.duration_p95: count=31, anomalies=4, earliest=2021-03-23 23:20:00 CST, "
    "max=0.7565s, p95=0.031175s, max_relative_deviation≈23.27 (very large duration spike).\n"
    "  2) MG02 — trace.from_Tomcat03.duration_p95: count=31, anomalies=4, earliest=2021-03-23 23:05:00 CST, "
    "max=5.1625s, p95=0.706375s, max_relative_deviation≈6.31.\n"
    "  3) Tomcat03 — trace.to_MG02.duration_p95: same pattern as #2 (count=31, anomalies=4, earliest=2021-03-23 23:05:00 CST), max_relative_deviation≈6.31.\n"
    "  4) MG02 — trace.from_dockerA1.duration_p95: count=26, anomalies=4, earliest=2021-03-23 23:00:00 CST, "
    "max=0.76275s, p95=0.11865s, max_relative_deviation≈5.43.\n"
    "  5) MG02 — trace.from_dockerA1.duration_mean: count=26, anomalies=4, earliest=2021-03-23 23:00:00 CST, "
    "max=0.1755s, p95≈0.03011s, max_relative_deviation≈4.83.\n\n"
    "• Overall patterns and interpretation:\n"
    "  - Most top anomalies are high-side duration spikes (trace durations well above historical p95).\n"
    "  - Affected traces are often inter-service paths (e.g., Tomcat ↔ MG02, dockerA/B → MG02), indicating latency spikes in service-to-service calls.\n"
    "  - Earliest anomalies cluster early in the window (≈23:00–23:06), with some later spikes (e.g., Tomcat04 at 23:20).\n\n"
    "If you want, I can provide the full top-20 table rows (cmdb_id, trace_name, counts, p95/p5, earliest anomaly timestamps, and relative deviations) for further inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(168,      cmdb_id                         trace_name  count_in_window  anomalies_count         earliest_anom_ts  max_value_in_window  min_value_in_window       p95        p5 direction  max_relative_deviation
0   Tomcat04            trace.self.duration_p95               31                4  2021-03-23 23:20:00 CST             0.756500             0.017550  0.031175  0.020200      high               23.266239
1       MG02   trace.from_Tomcat03.duration_p95               31                4  2021-03-23 23:05:00 CST             5.162500             0.020250  0.706375  0.028225      high                6.308441
2   Tomcat03         trace.to_MG02.duration_p95               31                4  2021-03-23 23:05:00 CST             5.162500             0.020250  0.706375  0.028225      high                6.308441
3       MG02   trace.from_dockerA1.duration_p95               26                4  2021-03-23 23:00:00 CST             0.762750             0.006000  0.118650  0.006850      high                5.428571
4       MG02  trace.from_dockerA1.duration_mean               26                4  2021-03-23 23:00:00 CST             0.175500             0.005800  0.030107  0.006083      high                4.829161
5   Tomcat04           trace.self.duration_mean               31                4  2021-03-23 23:02:00 CST             0.169000             0.006483  0.029728  0.007468      high                4.684876
6       MG02  trace.from_dockerB1.duration_mean               30                4  2021-03-23 23:06:00 CST             0.151361             0.006800  0.031695  0.006954      high                3.775586
7       MG02   trace.from_dockerB2.duration_p95               30                5  2021-03-23 23:00:00 CST             0.350800             0.006000  0.073700  0.007000      high                3.759837
8       MG02  trace.from_Tomcat03.duration_mean               31                4  2021-03-23 23:05:00 CST             0.597774             0.013500  0.131062  0.019446      high                3.560984
9   Tomcat03        trace.to_MG02.duration_mean               31                4  2021-03-23 23:05:00 CST             0.597774             0.013500  0.131062  0.019446      high                3.560984
10      IG01    trace.to_Tomcat04.duration_mean               31                4  2021-03-23 23:03:00 CST             1.019000             0.038800  0.224606  0.049552      high                3.536833
11  Tomcat04      trace.from_IG01.duration_mean               31                4  2021-03-23 23:03:00 CST             1.019000             0.038800  0.224606  0.049552      high                3.536833
12      MG01            trace.self.duration_p95               31                4  2021-03-23 23:01:00 CST             1.076000             0.005000  0.259775  0.034875      high                3.142046
13      MG02   trace.from_dockerA2.duration_p95               29                4  2021-03-23 23:04:00 CST             2.348700             0.007000  0.583270  0.007400      high                3.026780
14      MG01     trace.to_dockerA2.duration_p95               30                4  2021-03-23 23:02:00 CST             0.229400             0.025250  0.057332  0.025725      high                3.001221
15      MG01   trace.from_dockerB1.duration_p95               30                6  2021-03-23 23:00:00 CST             0.129500             0.009000  0.033382  0.009000      high                2.879278
16      MG01  trace.from_dockerB1.duration_mean               30                4  2021-03-23 23:13:00 CST             0.111563             0.006944  0.029789  0.007095      high                2.745051
17      MG02           trace.self.duration_mean               31                4  2021-03-23 23:06:00 CST             0.239910             0.017652  0.067276  0.020620      high                2.566030
18      MG02            trace.self.duration_p95               31                4  2021-03-23 23:06:00 CST             1.065900             0.034750  0.299975  0.040400      high                2.553296
19      IG02     trace.to_Tomcat03.duration_p95               31                4  2021-03-23 23:01:00 CST             4.971450             0.027250  1.416500  0.040875      high                2.509672)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse variables if available
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Components of interest for this step
target_components = ['MG02','MG01','Tomcat03','Tomcat04','Tomcat01','Tomcat02']

# Ensure numeric values and tz-aware timestamps
df_logs = log.copy()
df_logs['value'] = pd.to_numeric(df_logs['value'], errors='coerce')
df_logs['ts'] = pd.to_datetime(df_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global thresholds (p95 and p5) per (cmdb_id, log_name) across entire file
#    Restrict threshold computation to target components (but use whole file rows for them)
thr_df = df_logs[df_logs['cmdb_id'].isin(target_components)].groupby(
    ['cmdb_id','log_name'], as_index=False
)['value'].agg(
    p95=lambda s: s.quantile(0.95),
    p5 =lambda s: s.quantile(0.05)
)

# 2) Filter log rows to the incident window (inclusive)
start = pd.Timestamp('2021-03-23 23:00:00', tz=tz)
end   = pd.Timestamp('2021-03-23 23:30:00', tz=tz)
df_window = df_logs[(df_logs['ts'] >= start) & (df_logs['ts'] <= end)]

# Further restrict to target components
df_window = df_window[df_window['cmdb_id'].isin(target_components)].copy()

# 3) Merge thresholds into window rows and flag anomalies
merged = df_window.merge(thr_df, on=['cmdb_id','log_name'], how='left', validate='m:1')

# Flag anomalies using global thresholds
merged['is_anom'] = (merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])

# Helper to get earliest anomaly timestamp per group
def earliest_anom_ts(sub):
    an = sub[sub['is_anom']]
    if an.empty:
        return None
    return an['ts'].min().strftime('%Y-%m-%d %H:%M:%S %Z')

# Aggregate per (cmdb_id, log_name) in the window
grp = merged.groupby(['cmdb_id','log_name'], as_index=False)
agg = grp.agg(
    count_in_window = ('value','count'),
    anomalies_count = ('is_anom','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# compute earliest_anom_ts by iterating groups
earliest_map = {}
for (c,l), sub in merged.groupby(['cmdb_id','log_name']):
    earliest_map[(c,l)] = earliest_anom_ts(sub)
agg['earliest_anom_ts'] = agg.apply(lambda r: earliest_map.get((r['cmdb_id'], r['log_name'])), axis=1)

# Compute direction and max_relative_deviation
def compute_dev(row):
    p95 = row['p95']
    p5  = row['p5']
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    # compute high and low deviations safely
    high_dev = np.nan
    low_dev = np.nan
    if pd.notna(p95) and p95 != 0:
        high_dev = (maxv - p95) / p95
    if pd.notna(p5) and p5 != 0:
        low_dev = (p5 - minv) / p5
    # negative or NaN treated as 0 (no breach) per prior logic
    high_dev = high_dev if (pd.notna(high_dev) and high_dev > 0) else (np.nan if pd.isna(high_dev) else 0.0)
    low_dev  = low_dev  if (pd.notna(low_dev)  and low_dev  > 0) else (np.nan if pd.isna(low_dev)  else 0.0)
    hi = high_dev if pd.notna(high_dev) else -np.inf
    lo = low_dev  if pd.notna(low_dev)  else -np.inf
    if hi >= lo:
        direction = 'high'
        max_rel = hi if hi != -np.inf else np.nan
    else:
        direction = 'low'
        max_rel = lo if lo != -np.inf else np.nan
    return pd.Series({'direction': direction, 'max_relative_deviation': max_rel})

devs = agg.apply(compute_dev, axis=1)
agg = pd.concat([agg, devs], axis=1)

# 4) Filter combos with anomalies_count > 0 and prepare top 20 by max_relative_deviation
anoms = agg[agg['anomalies_count'] > 0].copy()

# Select output columns and round numeric values for compactness
cols = ['cmdb_id','log_name','count_in_window','anomalies_count','earliest_anom_ts',
        'max_value_in_window','min_value_in_window','p95','p5','direction','max_relative_deviation']
anoms = anoms[cols]

# numeric rounding
for c in ['max_value_in_window','min_value_in_window','p95','p5','max_relative_deviation']:
    anoms[c] = pd.to_numeric(anoms[c], errors='coerce')
anoms[['p95','p5','max_relative_deviation']] = anoms[['p95','p5','max_relative_deviation']].round(6)
anoms[['max_value_in_window','min_value_in_window']] = anoms[['max_value_in_window','min_value_in_window']].round(6)

anoms_sorted = anoms.sort_values('max_relative_deviation', ascending=False).head(20)

total_anomalous_combos = anoms.shape[0]

# Final compact output
total_anomalous_combos, anoms_sorted.reset_index(drop=True)

```
Out[5]:
```
```python
summary = (
    "Log analysis summary (window 2021-03-23 23:00–23:30 CST):\n\n"
    "- Total (cmdb_id, log_name) combos with anomalies: 12.\n\n"
    "- Key findings:\n"
    "  * log.row_count on Tomcat nodes shows pronounced low-side anomalies:\n"
    "    - Tomcat01: count_in_window=31, anomalies=6, earliest anomaly 23:03, max≈103, p95≈99, max_rel≈0.984\n"
    "    - Tomcat02: count_in_window=31, anomalies=5, earliest anomaly 23:00, max≈113, p95≈101, max_rel≈0.983\n"
    "    - Tomcat03: count_in_window=31, anomalies=4, earliest anomaly 23:02, max≈109, p95≈103.5, max_rel≈0.969\n"
    "    - Tomcat04: count_in_window=30, anomalies=5, earliest anomaly 23:06, max≈104, p95≈102.65, max_rel≈0.111\n"
    "    These indicate row_count dropped below historical lower thresholds (low-side breaches) for several Tomcat instances.\n\n"
    "  * log.gc_pause_max shows a few small anomalies across Tomcat nodes (mixed high/low), but relative deviations are small (≤0.2).\n\n"
    "  * log.error_count is flagged as anomalous for all four Tomcat nodes (anomalies_count equals full-window samples), "
    "but p95 and p5 are 0 and values are constant 0 — this yields no meaningful relative deviation (NaN) and is not actionable by magnitude.\n\n"
    "- Overall interpretation: The dominant and actionable signal is the low-side drop in log.row_count on Tomcat01–03 (and to a lesser extent Tomcat04). "
    "GC pause changes are minor; error_count is constant-zero and not informative for magnitude-based anomaly scoring."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(12,      cmdb_id          log_name  count_in_window  anomalies_count         earliest_anom_ts  max_value_in_window  min_value_in_window       p95       p5 direction  max_relative_deviation
0   Tomcat01     log.row_count               31                6  2021-03-23 23:03:00 CST               103.00                 1.00   99.0000  61.5000       low                0.983740
1   Tomcat02     log.row_count               31                5  2021-03-23 23:00:00 CST               113.00                 1.00  101.0000  58.0000       low                0.982759
2   Tomcat03     log.row_count               31                4  2021-03-23 23:02:00 CST               109.00                 2.00  103.5000  64.0000       low                0.968750
3   Tomcat02  log.gc_pause_max                4                3  2021-03-23 23:14:00 CST                 0.10                 0.03    0.1000   0.0375       low                0.200000
4   Tomcat04     log.row_count               30                5  2021-03-23 23:06:00 CST               104.00                56.00  102.6500  63.0000       low                0.111111
5   Tomcat03  log.gc_pause_max                4                2  2021-03-23 23:16:00 CST                 0.06                 0.02    0.0555   0.0215      high                0.081081
6   Tomcat04  log.gc_pause_max                6                4  2021-03-23 23:04:00 CST                 0.08                 0.06    0.0800   0.0625       low                0.040000
7   Tomcat01  log.gc_pause_max                4                2  2021-03-23 23:04:00 CST                 0.07                 0.05    0.0685   0.0515       low                0.029126
8   Tomcat01   log.error_count               31               31  2021-03-23 23:00:00 CST                 0.00                 0.00    0.0000   0.0000      high                     NaN
9   Tomcat02   log.error_count               31               31  2021-03-23 23:00:00 CST                 0.00                 0.00    0.0000   0.0000      high                     NaN
10  Tomcat03   log.error_count               31               31  2021-03-23 23:00:00 CST                 0.00                 0.00    0.0000   0.0000      high                     NaN
11  Tomcat04   log.error_count               30               30  2021-03-23 23:00:00 CST                 0.00                 0.00    0.0000   0.0000      high                     NaN)```
```